# Grounded Answers with DPO: Teaching LFM3-1.7B to Say "I Don't Know" with TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_dpo_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

Small models are usually deployed with retrieval: your app finds a few relevant documents, and the model answers from them. The main risk is a confident answer that isn't in the documents. This notebook trains [LFM3-1.7B](https://huggingface.co/LiquidAI/LFM3-1.7B) to answer only from the documents, and to reply with a fixed fallback sentence when the answer isn't there, using Direct Preference Optimization (DPO) with [TRL](https://github.com/huggingface/trl) and LoRA.

DPO learns from pairs of answers to the same prompt: a **chosen** answer and a **rejected** one. Here, the rejected answers are the model's own mistakes, sampled from LFM3-1.7B itself: hallucinated answers to questions the documents can't answer, wrong answers, and unnecessary refusals. The chosen answers are verified answers, or the exact fallback sentence. Training on its own mistakes targets exactly the behavior we want to change.

## 🎯 What you'll find

- **Grounded QA data**: fictional documents in 16 languages, with answerable and tricky unanswerable questions
- **LoRA + DPO**: the base model with the adapter disabled serves as the reference, so only one copy of the weights is in memory
- **Evaluation**: accuracy on answerable questions and on unanswerable ones, before and after training

# 📦 Installation & Setup

We pin every library to the versions this notebook was tested with.

In [ ]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

In [ ]:
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B is released in bfloat16. GPUs without bfloat16 support, like the T4, load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Loading the model from Transformers 🤗

LFM3-1.7B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="left")

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)
print("✅ Model loaded successfully!")

# 📚 Load the grounded QA dataset

Each example has a few short documents about fictional companies, products, places, and policies (so the model can't answer from memory), a question, and the gold label. About half of the questions can't be answered from the documents, and most of those are near-misses: the documents give the price of a different plan, the start date but not the end date, or the policy for a neighboring case.

- `train`: preference pairs, with a `chosen` and a `rejected` answer for each prompt
- `test`: questions with their gold answers, for evaluation
- `test_unseen`: questions about domains that never appear in training
- `showcase`: a few hand-checked examples from those unseen domains

In [ ]:
from datasets import load_dataset

dataset = load_dataset("LiquidAI/grounded-qa")
print(dataset)

example = dataset["train"][0]
print(f"\n❓ Question ({example['language']}): {example['question']}")
print(f"📎 Answerable: {example['answerable']}")
print(f"\n✅ Chosen: {example['chosen']}")
print(f"\n❌ Rejected: {example['rejected']}")

## Build the prompts

The instructions and the fallback sentence go in the system prompt, in the language of the question. The documents and the question go in the user message.

We use LFM3's non-reasoning mode (`enable_thinking=False`), which answers right away. TRL reads the `chat_template_kwargs` column and passes it to the chat template.

In [ ]:
SYSTEM_PROMPT = (
    "Answer the user's question using only the documents they provide. Answer in the language of the question, "
    "in one or two sentences. If the documents don't contain the answer, reply with exactly this sentence and "
    "nothing else: {fallback}"
)

def build_messages(example):
    documents = "\n\n".join(f"[{i}] {doc['title']}\n{doc['text']}" for i, doc in enumerate(example["documents"], start=1))
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(fallback=example["fallback"])},
        {"role": "user", "content": f"Documents:\n\n{documents}\n\nQuestion: {example['question']}"},
    ]

def to_preference(example):
    return {
        "prompt": build_messages(example),
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
        "chat_template_kwargs": {"enable_thinking": False},
    }

train_dataset = dataset["train"].map(to_preference, remove_columns=dataset["train"].column_names)
print(train_dataset[0]["prompt"][1]["content"][:1200])

# 📏 Evaluate the model before training

We generate an answer for each test question, with greedy decoding and the repetition penalty recommended in the model card (`repetition_penalty=1.1`), and check it against the rules of the system prompt:

- **Answerable questions**: the answer is correct if it contains the gold answer and doesn't include the fallback sentence. We also count how often the model declines them, which should stay low.
- **Unanswerable questions**: the answer is correct if it's exactly the fallback sentence, as the system prompt asks. The original model often writes the fallback sentence and then answers anyway, which an app can't rely on.

We evaluate on 200 questions from `test` (the training domains) and 200 from `test_unseen` (domains that never appear in training), to check that the behavior generalizes.

In [ ]:
from tqdm.auto import tqdm

def normalize(text):
    return " ".join(text.casefold().replace("’", "'").split()).strip(" .!?。")

@torch.no_grad()
def evaluate(model, split, n=200, batch_size=16):
    model.eval()
    examples = dataset[split].shuffle(seed=42).select(range(n))
    results = {"answerable": [], "unanswerable": [], "over_declined": []}
    for start in tqdm(range(0, n, batch_size), desc=f"Evaluating {split}"):
        batch = examples.select(range(start, min(start + batch_size, n)))
        prompts = [tokenizer.apply_chat_template(build_messages(e), add_generation_prompt=True, tokenize=False, enable_thinking=False) for e in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)
        for output, example in zip(outputs[:, inputs["input_ids"].shape[1]:], batch):
            answer = normalize(tokenizer.decode(output, skip_special_tokens=True))
            fell_back = answer == normalize(example["fallback"])
            if example["answerable"]:
                results["answerable"].append(normalize(example["fallback"]) not in answer and normalize(example["answer"]) in answer)
                results["over_declined"].append(fell_back)
            else:
                results["unanswerable"].append(fell_back)
    return {kind: sum(r) / len(r) for kind, r in results.items()}

def show(results, title):
    print(title)
    for split, r in results.items():
        print(f"   {split:<12} answerable: answered correctly {r['answerable']:.1%}, declined {r['over_declined']:.1%} | unanswerable: declined {r['unanswerable']:.1%}")

baseline = {split: evaluate(model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")

# 🎯 Direct Preference Optimization (DPO + LoRA)

## Wrap the model with PEFT

We adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # optional: LFM3's per-head attention gate
    ],
    task_type="CAUSAL_LM",
)

## Launch training

`beta` controls how far the model can move away from the reference: lower values allow larger changes.

We combine the DPO loss with an SFT loss on the chosen answers (`loss_type=["sigmoid", "sft"]`). DPO alone only learns which answer is *better*: here it quickly stops the model from prefixing its answers with the fallback sentence, but it also pushes the model away from the fallback when it should use it. The SFT term teaches the model to actually write the chosen answers, including the exact fallback sentence. In our runs, it took the share of declined unanswerable questions from about 20% to about 90%.

During training, watch `rewards/accuracies` (how often the model prefers the chosen answer) and `rewards/margins` go up.

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir="./LFM3-1.7B-grounded-dpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # <- effective batch size of 16 pairs
    learning_rate=2e-5,
    loss_type=["sigmoid", "sft"],  # <- DPO loss + SFT loss on the chosen answers
    loss_weights=[1.0, 1.0],
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # <- a float is a fraction of the total steps
    beta=0.1,
    max_length=3072,
    logging_steps=10,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

print("🚀 Starting DPO training...")
trainer.train()
print("🎉 DPO training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

# 📊 Evaluate the trained model

In [ ]:
final = {split: evaluate(trainer.model, split) for split in ["test", "test_unseen"]}
show(baseline, "📏 Before training")
show(final, "📏 After training")

# 🔍 Original vs fine-tuned model

To see what changed, we ask the original model and the trained model the same questions. With LoRA, `disable_adapter()` temporarily turns the adapter off, so we don't need to load the original model again. The `showcase` split holds hand-checked questions about domains the model never saw during training.

In [ ]:
def answer(model, example):
    prompt = tokenizer.apply_chat_template(build_messages(example), add_generation_prompt=True, tokenize=False, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False, repetition_penalty=1.1)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def verdict(text, example):
    fell_back = normalize(text) == normalize(example["fallback"])
    if example["answerable"]:
        return "✅" if normalize(example["fallback"]) not in normalize(text) and normalize(example["answer"]) in normalize(text) else "❌"
    return "✅" if fell_back else "❌"

trainer.model.eval()
for example in dataset["showcase"]:
    with trainer.model.disable_adapter():
        original = answer(trainer.model, example)
    trained = answer(trainer.model, example)
    print(f"\n❓ {example['question']} ({example['language']})")
    print(f"   🎯 Gold:       {example['reference_answer']}")
    print(f"   {verdict(original, example)} Original:   {original}")
    print(f"   {verdict(trained, example)} Fine-tuned: {trained}")

And the summary over all the evaluation questions:

In [ ]:
print(f"{'':<46}{'original':>10}{'fine-tuned':>12}")
for split, name in [("test", "training domains"), ("test_unseen", "unseen domains")]:
    for kind, label in [("answerable", "answerable, answered correctly"), ("over_declined", "answerable, declined (lower is better)"), ("unanswerable", "unanswerable, declined")]:
        print(f"{name + ', ' + label:<46}{baseline[split][kind]:>10.1%}{final[split][kind]:>12.1%}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-1.7B-grounded-dpo-lora")

output_dir = "./LFM3-1.7B-grounded-dpo-merged"
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# 🚀 Next steps

- **Your own documents**: sample answers from your model on your own retrieval prompts, label them (by hand, with code, or with a stronger model as a judge), and pair a good answer with a bad one for each prompt.
- **Keep it on-policy**: after a round of DPO, sample again from the trained model and build new pairs from its remaining mistakes.